# DentVLM panoramic pipeline (Kaggle runner)

One image goes through the paper's own protocol; every question is one DentVLM was trained on:

```text
panoramic X-ray
   -> 13 yes/no questions, one per DentVLM panoramic task (Supplementary Table 7 wording), whole image
   -> line 1 of each reply = Yes/No; the rationale names the location with the nine trained
      descriptors -> six dental-arch cells; multiplicity = number of cells named
   -> optional knobs: 3-phrasing vote, out-of-distribution count question, crop comparison
   -> JSON per image, deterministic dentist summary, TP/FP/TN/FN evaluation
   -> location truth: ground-truth boxes are translated into the same six cells by a vision LLM
      (numbered boxes drawn on the image -> FDI quadrant x anterior/posterior), by DentVLM itself
      (experimental), or by fixed windows
```

Run the cells in order. Edit only **CELL 3**. DentVLM is published as bf16 safetensors on a
gated Hugging Face repo (automatic approval): accept the license once, store a token as a
Kaggle secret, and let CELL 6 convert it to GGUF the first time (keep the two files).

In [ ]:
# ============================================================
# CELL 1 - Python dependencies (llama.cpp is compiled later with CUDA)
# ============================================================
%pip install -q "huggingface_hub>=0.26" "openai>=1.55" "requests>=2.31" "pillow>=10.0" "pandas>=1.5"
print("Python dependencies installed.")

In [ ]:
# ============================================================
# CELL 2 - Locate and import the project files
# ============================================================
import os, sys, json, shutil, subprocess
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/dental_x-ray")  # folder holding the four .py files below
REQUIRED_PROJECT_FILES = {"dental_pipeline.py", "dental_eval.py", "llama_runtime.py", "location_adapter.py"}
if not PROJECT_DIR.is_dir() and Path.cwd().joinpath("dental_pipeline.py").is_file():
    PROJECT_DIR = Path.cwd()
missing = [f for f in REQUIRED_PROJECT_FILES if not (PROJECT_DIR / f).is_file()]
if missing:
    raise FileNotFoundError(f"Missing project files in {PROJECT_DIR}: {missing}")
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import dental_pipeline as dp
import dental_eval as ev
import location_adapter as la
from llama_runtime import LlamaCppServer, build_llama_cpp, convert_to_gguf, download_gguf, local_gguf
print("PROJECT_DIR =", PROJECT_DIR)
print("Project imports succeeded.")

In [ ]:
# ============================================================
# CELL 3 - CONFIGURATION (the only cell to edit between experiments)
# ============================================================
OUTPUT_DIR = "/kaggle/working/dental_outputs"

# Model backend: local DentVLM through llama.cpp, or an OpenAI-compatible vision API for comparison.
BACKEND = "api"  # "local" | "api"
API = {"base_url": "https://openrouter.ai/api/v1", "api_key_env": "OPENROUTER_API_KEY",
       "model": "qwen/qwen3-vl-235b-a22b-thinking",
       "request_options": {"extra_body": {"provider": {
           "only": ["NovitaAI"], "allow_fallbacks": False}}}}

# Model files. DentVLM (Hugging Face ZJU-AI4H/DentVLM, gated with automatic approval, CC BY-NC 4.0)
# ships as bf16 safetensors; the pipeline runs a GGUF conversion of it. MODEL_SOURCE:
#   "convert" - first run: download the checkpoint and convert it into MODEL_DIR (~17 GB scratch under
#               CONVERT_WORK_DIR, Q8_0 language model + f16 vision projector, ~9.5 GB kept)
#   "local"   - use the converted files already in MODEL_DIR (e.g. an attached Kaggle dataset)
#   "hf"      - download MODEL_FILENAME / MMPROJ_FILENAME from your own GGUF_REPO_ID
MODEL_SOURCE = "convert"
MODEL_DIR = "/kaggle/working/models/dentvlm"
MODEL_FILENAME = "DentVLM-Q8_0.gguf"
MMPROJ_FILENAME = "DentVLM-mmproj-f16.gguf"
GGUF_REPO_ID = "REPLACE/DentVLM-GGUF"
CONVERT_WORK_DIR = "/tmp/dentvlm_hf"  # scratch for the 17 GB safetensors; not persisted
HF_TOKEN_SECRET = "HF_TOKEN"  # Kaggle secret holding a Hugging Face token (or set env HF_TOKEN)

# llama.cpp runtime. Context must hold image tokens + question + answer (the authors' max input: 16384).
LLAMA_CPP_DIR = "/kaggle/working/llama.cpp"
LLAMA_CPP_REF = "b10516"
SERVER_HOST, SERVER_PORT, SERVER_ALIAS = "127.0.0.1", 8080, "dentvlm"
SERVER_LOG_PATH = "/kaggle/working/llama_dentvlm_server.log"
N_GPU_LAYERS = 999
CTX_SIZE = 16384
IMAGE_MAX_TOKENS = 8192   # = the authors' max_pixels 8192x28x28; 1369 reproduces their 1024x1024 ablation bound
IMAGE_MIN_TOKENS = None   # no floor, as in the authors' inference (min 4 tokens)
CUDA_ARCH = None          # None = auto-detect (P100 fallback 60)
BUILD_JOBS = 4
SERVER_STARTUP_TIMEOUT = 300.0

# Generation: one greedy attempt per question, the authors' output cap of 512 tokens.
MAX_TOKENS = 512
TEMPERATURE = 0.0
CACHE_PROMPT = True       # reuse the image KV prefix across the questions of one image
REQUEST_TIMEOUT_SECONDS = 600.0

# Protocol. Defaults are the paper's protocol; see dental_pipeline.Protocol for each knob.
PROTOCOL = dp.Protocol(
    phrasings=1,            # 3 = ask three verbatim wordings per task and vote (in-distribution ensemble)
    region_vote="union",    # with phrasings=3: "union" (matching voting) or "majority"
    location="rationale",   # "rationale" (free, in-distribution) | "crops" (comparison only) | "none"
    count_question=False,   # True = out-of-distribution tooth-count question for positive findings
    ask_untrained=False,    # True = also ask the five UMFIH classes DentVLM has no task for
    extra_tasks=True,       # residual crown, eruption space, calculus: reported, not scored
)
SMOKE_IMAGES = 3

# Location truth: how ground-truth boxes are translated into the six cells DentVLM names, for scoring.
#   "llm"      - numbered boxes drawn on the radiograph, a strong vision LLM (OpenAI-compatible API) returns
#                FDI quadrant x anterior/posterior per box; mapped onto the cells in Python. Recommended.
#   "fdm"      - DentVLM itself, one Table S7 question per spotlighted box (experimental; needs BACKEND="local")
#   "geometry" - fixed cell windows, no model calls (the previous behaviour)
LOCATION_TRUTH = "llm"
ADAPTER_API = {
    "base_url": "https://api.openai.com/v1", "api_key_env": "OPENAI_API_KEY", "model": "gpt-5",
    "token_param": "max_completion_tokens",  # "max_tokens" for non-reasoning models (Gemini compat endpoint, NIM, ...)
    "temperature": None,                     # reasoning models reject one; 0.0 for others
    "max_output_tokens": 8192, "max_boxes_per_call": 12, "request_options": {},
}
ADAPTER_FDM_MARGIN = 0.06  # spotlight margin around the box, fraction of the image size

# Datasets to run and score. kind "yolo": UMFIH 14-class layout; kind "dentex": DENTEX split.
# "limit" runs only the first N images (sorted by id); None = all.
DATASETS = [
    {"name": "umfih_test", "kind": "yolo", "limit": None,
     "images": "/kaggle/working/umfih_14class/data/test/images",
     "labels": "/kaggle/working/umfih_14class/data/test/labels"},
    # {"name": "umfih_external", "kind": "yolo", "limit": None,
    #  "images": "/kaggle/working/umfih_14class/external/images",
    #  "labels": "/kaggle/working/umfih_14class/external/labels"},
    # DENTEX: fully labeled train split (705 images, COCO-style JSON) and/or the 50-image validation split
    # (validation_triple.json). DentVLM's authors used only the 242-image official test split for their
    # external validation, so both of these are held-out for the model as well.
    # {"name": "dentex_train", "kind": "dentex", "limit": None,
    #  "images": "/kaggle/working/DENTEX/training_data/quadrant-enumeration-disease/xrays",
    #  "annotations": "/kaggle/working/DENTEX/training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json"},
    # {"name": "dentex_val", "kind": "dentex", "limit": None,
    #  "images": "/kaggle/working/DENTEX/validation_data/quadrant_enumeration_disease/xrays",
    #  "annotations": "/kaggle/working/DENTEX/validation_triple.json"},
]

NEEDS_LOCAL_RUNTIME = BACKEND == "local"
print("backend =", BACKEND, "| model source =", MODEL_SOURCE, "| protocol =", PROTOCOL,
      "| location truth =", LOCATION_TRUTH, "| datasets =", [d["name"] for d in DATASETS])

In [ ]:
# ============================================================
# CELL 4 - Environment diagnostics
# ============================================================
import platform
print("Python:", platform.python_version(), "|", platform.platform())


def detect_cuda_arch(fallback="60"):
    try:
        output = subprocess.check_output(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                                         text=True, stderr=subprocess.STDOUT)
        arch = output.strip().splitlines()[0].strip().replace(".", "")
        if arch.isdigit():
            return arch
    except Exception as exc:
        print("CUDA architecture auto-detection failed:", exc)
    print(f"Falling back to CUDA architecture {fallback}.")
    return fallback


CUDA_ARCH_RESOLVED = None
if NEEDS_LOCAL_RUNTIME:
    for executable in ("git", "cmake", "nvcc", "nvidia-smi"):
        print(f"{executable:12s}:", shutil.which(executable))
    if shutil.which("nvidia-smi"):
        subprocess.run(["nvidia-smi"], check=False)
    for executable in ("cmake", "git", "nvcc"):
        if not shutil.which(executable):
            raise RuntimeError(f"{executable} is required to build llama.cpp; enable a GPU accelerator.")
    CUDA_ARCH_RESOLVED = str(CUDA_ARCH) if CUDA_ARCH else detect_cuda_arch("60")
    print("CUDA_ARCH_RESOLVED =", CUDA_ARCH_RESOLVED)
else:
    print("API backend selected; local checks skipped.")

In [ ]:
# ============================================================
# CELL 5 - Build or find the pinned llama.cpp server
# ============================================================
LLAMA_SERVER = None
if NEEDS_LOCAL_RUNTIME:
    # Reuses an existing build only if it was built from LLAMA_CPP_REF; otherwise rebuilds.
    LLAMA_SERVER = Path(build_llama_cpp(source_dir=LLAMA_CPP_DIR, cuda_arch=CUDA_ARCH_RESOLVED,
                                        jobs=BUILD_JOBS, ref=LLAMA_CPP_REF)).resolve()
    print("llama-server =", LLAMA_SERVER)
else:
    print("API backend selected; llama.cpp build skipped.")

In [ ]:
# ============================================================
# CELL 6 - DentVLM GGUF files: convert once, then reuse
# ============================================================
MODEL_PATH = MMPROJ_PATH = None
if NEEDS_LOCAL_RUNTIME:
    hf_token = os.environ.get("HF_TOKEN")
    if not hf_token:
        try:
            hf_token = __import__("kaggle_secrets").UserSecretsClient().get_secret(HF_TOKEN_SECRET)
        except Exception as exc:
            print(f"No Hugging Face token available ({exc}); only MODEL_SOURCE='local' works without one.")
    if MODEL_SOURCE == "convert":
        files = convert_to_gguf(MODEL_DIR, LLAMA_CPP_DIR, hf_token=hf_token, work_dir=CONVERT_WORK_DIR,
                                model_filename=MODEL_FILENAME, mmproj_filename=MMPROJ_FILENAME)
        print("Keep the two files above (private Kaggle dataset or Hugging Face repo) and switch "
              "MODEL_SOURCE to 'local' or 'hf' for the next session.")
    elif MODEL_SOURCE == "local":
        files = local_gguf(MODEL_DIR, MODEL_FILENAME, MMPROJ_FILENAME)
    elif MODEL_SOURCE == "hf":
        files = download_gguf(MODEL_DIR, GGUF_REPO_ID, MODEL_FILENAME, MMPROJ_FILENAME, hf_token)
    else:
        raise ValueError(f"unknown MODEL_SOURCE {MODEL_SOURCE!r}")
    MODEL_PATH, MMPROJ_PATH = Path(files.model_path).resolve(), Path(files.mmproj_path).resolve()
    for label, path in (("Language model", MODEL_PATH), ("Vision projector", MMPROJ_PATH)):
        print(f"{label}: {path} ({path.stat().st_size / 1024**3:.2f} GiB)")
else:
    print("API backend selected; model files skipped.")

In [ ]:
# ============================================================
# CELL 7 - Start a clean DentVLM llama.cpp server
# ============================================================
import requests

previous = globals().get("server")
server = None
if NEEDS_LOCAL_RUNTIME:
    if previous is not None:
        try:
            previous.stop()
        except Exception as exc:
            print("Previous server cleanup:", exc)
    server = LlamaCppServer(binary=LLAMA_SERVER, model_path=MODEL_PATH, mmproj_path=MMPROJ_PATH,
                            host=SERVER_HOST, port=SERVER_PORT, alias=SERVER_ALIAS, n_gpu_layers=N_GPU_LAYERS,
                            ctx_size=CTX_SIZE, image_max_tokens=IMAGE_MAX_TOKENS, image_min_tokens=IMAGE_MIN_TOKENS,
                            startup_timeout=SERVER_STARTUP_TIMEOUT, log_path=SERVER_LOG_PATH)
    server.start(reuse_existing=False)
    ids = [m.get("id") for m in requests.get(f"{server.base_url}/v1/models", timeout=10).json().get("data", [])]
    if SERVER_ALIAS not in ids:
        raise RuntimeError(f"Expected alias {SERVER_ALIAS!r}; /v1/models returned {ids}. See {SERVER_LOG_PATH}.")
    print("DentVLM server verified at", server.base_url)
else:
    print("API backend selected; server startup skipped.")

In [ ]:
# ============================================================
# CELL 8 - Model runner (one image + one question -> one answer)
# ============================================================
if NEEDS_LOCAL_RUNTIME:
    runner = dp.VisionRunner(base_url=f"{server.base_url}/v1", api_key="local-llama-cpp", model=SERVER_ALIAS,
                             max_tokens=MAX_TOKENS, temperature=TEMPERATURE, timeout=REQUEST_TIMEOUT_SECONDS,
                             local=True, cache_prompt=CACHE_PROMPT)
else:
    api_key = os.environ.get(API["api_key_env"]) or __import__("kaggle_secrets").UserSecretsClient().get_secret(API["api_key_env"])
    runner = dp.VisionRunner(base_url=API["base_url"], api_key=api_key, model=API["model"], max_tokens=MAX_TOKENS,
                             temperature=TEMPERATURE, timeout=REQUEST_TIMEOUT_SECONDS, local=False,
                             request_options=API.get("request_options"))
print("Runner ready:", runner.settings())

In [ ]:
# ============================================================
# CELL 9 - Load ground truth for every dataset (no model calls)
# ============================================================
GT = {}
for spec in DATASETS:
    if spec["kind"] == "yolo":
        gt = ev.load_yolo(spec["images"], spec["labels"])
    elif spec["kind"] == "dentex":
        gt = ev.load_dentex(spec["images"], spec["annotations"])
    else:
        raise ValueError(f"unknown dataset kind {spec['kind']!r}")
    if spec.get("limit"):
        gt = dict(sorted(gt.items())[: spec["limit"]])
    GT[spec["name"]] = gt
    positives = sum(1 for g in gt.values() if g["boxes"])
    print(f"{spec['name']}: {len(gt)} images, {positives} with at least one finding, "
          f"{sum(len(g['boxes']) for g in gt.values())} boxes")

In [ ]:
# ============================================================
# CELL 10 - Smoke test: raw replies on a few images (no probe needed)
# ============================================================
# Sends the primary implant and caries questions to the first SMOKE_IMAGES images. Expect line 1 to be
# Yes/No on every reply, a location descriptor on most Yes replies, and no truncation at MAX_TOKENS.
first = next(iter(GT.values()))
smoke_paths = [g["path"] for _, g in sorted(first.items())[:SMOKE_IMAGES]]
rows = []
for path in smoke_paths:
    for task in ("implant", "caries"):
        reply = runner.ask(path, dp.questions_for(task)[0])
        row = {"image": Path(path).name, "task": task, "answer": dp.extract_answer(reply["text"]),
               "regions": dp.extract_regions(reply["text"]), "truncated": reply["truncated"], "text": reply["text"]}
        rows.append(row)
        print(f"--- {row['image']} | {task} | answer={row['answer']} | regions={row['regions']} | "
              f"truncated={row['truncated']}\n{reply['text'][:600]}\n")
yes_rows = [r for r in rows if r["answer"] == "yes"]
print(f"parsed line-1 answers: {sum(r['answer'] is not None for r in rows)}/{len(rows)} | "
      f"yes replies naming a region: {sum(bool(r['regions']) for r in yes_rows)}/{len(yes_rows)} | "
      f"truncated: {sum(r['truncated'] for r in rows)}")
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_DIR, "smoke.json").write_text(json.dumps(rows, indent=1, ensure_ascii=False), encoding="utf-8")

In [ ]:
# ============================================================
# CELL 11 - Run every dataset (resumable: finished images are skipped)
# ============================================================
# Provenance is hashed into the run manifest, so a changed checkpoint or image budget cannot be
# silently mixed into a resumed run.
PROVENANCE = ({"model_file": MODEL_FILENAME, "mmproj_file": MMPROJ_FILENAME, "llama_cpp_ref": LLAMA_CPP_REF,
               "ctx_size": CTX_SIZE, "image_max_tokens": IMAGE_MAX_TOKENS, "image_min_tokens": IMAGE_MIN_TOKENS}
              if NEEDS_LOCAL_RUNTIME else {"api_model": API["model"], "base_url": API["base_url"],
                                           "request_options": API.get("request_options", {})})
RUN_DIRS = {}
for spec in DATASETS:
    images = {image_id: g["path"] for image_id, g in GT[spec["name"]].items()}
    RUN_DIRS[spec["name"]] = dp.run_dataset(runner, images, Path(OUTPUT_DIR) / spec["name"],
                                            protocol=PROTOCOL, resume=True, provenance=PROVENANCE)
    print("Saved:", RUN_DIRS[spec["name"]])

In [ ]:
# ============================================================
# CELL 12 - Location truth: translate ground-truth boxes into DentVLM's six cells (resumable)
# ============================================================
# Runs once per dataset and adapter (independent of the model run; can be re-used across protocols).
# One JSON per image under <dataset>/location_truth/<adapter>/boxes, drawn images under .../drawn for audit.
# Boxes the adapter cannot place fall back to the fixed windows and are counted as such.
ADAPTED = {}
if LOCATION_TRUTH == "llm":
    adapter_key = os.environ.get(ADAPTER_API["api_key_env"]) or __import__("kaggle_secrets").UserSecretsClient().get_secret(ADAPTER_API["api_key_env"])
    adapter = la.LLMAdapter(base_url=ADAPTER_API["base_url"], api_key=adapter_key, model=ADAPTER_API["model"],
                            token_param=ADAPTER_API["token_param"], temperature=ADAPTER_API["temperature"],
                            max_output_tokens=ADAPTER_API["max_output_tokens"], max_boxes_per_call=ADAPTER_API["max_boxes_per_call"],
                            request_options=ADAPTER_API["request_options"])
elif LOCATION_TRUTH == "fdm":
    if not NEEDS_LOCAL_RUNTIME:
        raise RuntimeError("LOCATION_TRUTH='fdm' needs the local DentVLM server (BACKEND='local').")
    adapter = la.FdmAdapter(runner, margin=ADAPTER_FDM_MARGIN)
elif LOCATION_TRUTH == "geometry":
    adapter = None
else:
    raise ValueError(f"unknown LOCATION_TRUTH {LOCATION_TRUTH!r}")

for spec in DATASETS:
    name = spec["name"]
    if adapter is None:
        print(f"{name}: location truth from fixed windows (and FDI tooth numbers where the dataset has them)")
        continue
    ADAPTED[name] = la.adapt_dataset(adapter, GT[name], Path(OUTPUT_DIR) / name / "location_truth" / adapter.name, resume=True)
    print(f"{name}: {la.summarize(ADAPTED[name])}")
    agreement = ev.truth_agreement(GT[name], ADAPTED[name])
    if agreement["boxes_with_fdi"]:
        # DENTEX carries FDI tooth numbers: exact cells, so this is the adapter's own accuracy against the fixed windows.
        print(f"{name}: adapter vs FDI truth {agreement}")


In [ ]:
# ============================================================
# CELL 13 - Evaluate: TP/FP/TN/FN per finding, regions, optional counts, per-image summaries
# ============================================================
import pandas as pd
from IPython.display import display

REPORTS = {}
for spec in DATASETS:
    name = spec["name"]
    results = dp.load_results(Path(OUTPUT_DIR) / name)
    truth = ev.apply_adapted(GT[name], ADAPTED[name]) if name in ADAPTED else GT[name]
    REPORTS[name] = ev.evaluate(truth, results, dataset=name, out_dir=Path(OUTPUT_DIR) / name / "evaluation")
    print(f"\n===== {name} =====")
    display(pd.DataFrame([REPORTS[name]["summary"]]).T)
    display(pd.DataFrame(REPORTS[name]["presence"]).set_index("condition"))
    if REPORTS[name]["counts"]:
        display(pd.DataFrame(REPORTS[name]["counts"]).set_index("condition"))
    if REPORTS[name]["regions"]:
        display(pd.DataFrame(REPORTS[name]["regions"]).set_index("condition"))
if len(REPORTS) > 1:
    print("\n===== pooled over shared findings =====")
    display(pd.DataFrame(ev.pooled_presence(list(REPORTS.values()))).set_index("condition"))
print("Presence rows use only findings the dataset annotates and the model was asked about; findings without a "
      "DentVLM task are listed under not_assessed. Unparseable answers are excluded from TP/FP/TN/FN and reported "
      "as counts; the per-image complete-case rate and recall are strict and count them as not caught. "
      "summary.location_truth says which method placed the true boxes for the region rows.")

In [ ]:
# ============================================================
# CELL 14 - Side convention check, then one image: dentist summary and raw answers
# ============================================================
# DentVLM's "left"/"right" follow Table S6 (its "left posterior" = FDI quadrants 1 and 4, the patient's right,
# which is the image left). Agreement far above 50% confirms dental_pipeline.LEFT_IS_IMAGE_LEFT. Far below means
# set LEFT_IS_IMAGE_LEFT = False in dental_pipeline.py, run `import importlib; importlib.reload(dp); importlib.reload(ev)`,
# and re-run CELL 13: no new model calls, the saved replies keep DentVLM's own words and the adapted truth is re-mapped.
for spec in DATASETS:
    print(spec["name"], "side agreement:", ev.side_agreement(GT[spec["name"]], dp.load_results(Path(OUTPUT_DIR) / spec["name"])))

INSPECT_DATASET = DATASETS[0]["name"]
INSPECT_IMAGE = None  # None = first image of the dataset
SHOW_RAW_CALLS = False

results = dp.load_results(Path(OUTPUT_DIR) / INSPECT_DATASET)
image_id = INSPECT_IMAGE or next(iter(sorted(results)))
result = results[image_id]
print()
print(dp.dentist_report(result))
truth = [b["condition"] for b in GT[INSPECT_DATASET][image_id]["boxes"]]
print("\nGround-truth boxes:", {c: truth.count(c) for c in dict.fromkeys(truth)} or "none")
if INSPECT_DATASET in ADAPTED:
    for k, record in enumerate(ADAPTED[INSPECT_DATASET][image_id]["boxes"], 1):
        print(f"  box {k}: {record['condition']} -> {record['regions']} ({record['source']}; fixed windows {record['geometry']})")
if SHOW_RAW_CALLS:
    for call in result["calls"]:
        print(f"\n--- {call['stage']} | {call['task']} | {call['cell']} | finish={call['finish_reason']}")
        print(call["text"][:800])